<a href="https://colab.research.google.com/github/cynthiiaa/neuralgcm-workshop/blob/main/AMLC_Jax_Primer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Author: Cythia Ukawu

# JAX Primer: A Crash Course for Data Professionals
**What you'll walk away with:**
- Enough JAX fluency to follow the main workshop
- Mental models that connect JAX concepts to tools you already know
- A cheat sheet for three important JAX operations: `grad`, `jit`, `vmap`


## [Link to Paper](https://arxiv.org/pdf/2311.07222)

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import time

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")
# If this says 'cpu', that's fine, it's just faster on GPU/TPU.

JAX version: 0.7.2
Available devices: [CpuDevice(id=0)]


# Part 1: What Is JAX and Why Should You Care?

**JAX is "differentiable" NumPy that you can run on GPUs or TPUs.**

It is heavily optimized for functional programming and scientific computing. If you know NumPy, you already know ~80% of JAX's API.

## Where JAX Fits in Your Toolbox

Example Stack:

| Task | Your current tool |
|------|------------------|
| Data wrangling | Pandas, SQL, PySpark |
| Array math | NumPy |
| Classical ML (regression, classification, etc) | sklearn, xgboost |
| Visualization | matplotlib, seaborn |

JAX replaces **NumPy** in situations where you need one or more of:

1. **Automatic Differentiation** - Jax can automatically calculate the exact derivatives (or gradients) of any mathematical function or algorithm written in the code. In machine learning, these gradients are necessary to update and train neural networks.
2. **Speed** - Your NumPy code is a bottleneck, and you want compiled performance without rewriting in C. For example, a type of bottleneck for these spectral weather models is calculating forward and inverse spherical harmonic transformations, which require massive matrix-matrix multiplications.
3. **GPU/TPU acceleration** - TPUs are extremely fast at computing matrix multiplication. JAX acts as the perfect software bridge to this hardware. By writing the physics engine in JAX, the researchers could use JAX's XLA SPMD compiler and a feature called `shard_map` to automatically distribute and parallelize these massive matrix multiplications across multiple TPU cores, splitting the workload across the x, y, and z spatial dimensions.

The key capability we'll fous on is  **automatic differentiation**. This feature is what allowed the Google researchers to compute derivatives through both the physics equations *and* the ML model simultaneously!

# Part 2: JAX Arrays

## NumPy arrays and JAX arrays look identical…

If you can do this in NumPy:

```python
import numpy as np
x = np.ones((3, 2))
x = x * 2 + 1
```

…you can do the same thing in JAX:

```python
import jax.numpy as jnp
x = jnp.ones((3, 2))
x = x * 2 + 1
```

## `jax.numpy` is “NumPy, but traceable.”

When you write a normal mathematical function in standard NumPy, the computer executes the math immediately and only gives you the final answer. It does not remember the step-by-step path it took to get there. `jax.numpy`, on the other hand, acts as a *recording device*. When you run your code, JAX does not just calculate the answer; it builds a "trace". This **trace** is a massive, step-by-step map (known as a computation graph) of every single addition, multiplication, and transformation used along the way.

By wrapping your code in specific JAX commands like `grad`/`jit`/`vmap`, you tell the system how to utilize that recorded trace.

This *traceability* is what makes the entire NeuralGCM architecture possible. By writing the dynamical core entirely in `jax.numpy`, the researchers ensured that every fluid dynamics equation left a **trail**. If the physics engine couldn't be traced, the neural network never learns how its small adjustments could snowball into massive errors throughout the physics engine over multiple days.


In [3]:
# NumPy
np_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"NumPy: {np_array}  (type: {type(np_array).__name__})")

# JAX: same syntax, different type under the hood
jax_array = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"JAX:   {jax_array}  (type: {type(jax_array).__name__})")

# All your familiar operations work
print(f"\nMean:  np={np.mean(np_array)}, jnp={jnp.mean(jax_array)}")
print(f"Slice: np={np_array[1:3]}, jnp={jax_array[1:3]}")
print(f"Math:  np={np.sin(np_array[:3])}, jnp={jnp.sin(jax_array[:3])}")

NumPy: [1. 2. 3. 4. 5.]  (type: ndarray)
JAX:   [1. 2. 3. 4. 5.]  (type: ArrayImpl)

Mean:  np=3.0, jnp=3.0
Slice: np=[2. 3.], jnp=[2. 3.]
Math:  np=[0.84147098 0.90929743 0.14112001], jnp=[0.84147096 0.9092974  0.14112   ]


## JAX follows functional programming (no in-place modification)

In NumPy, you do this all the time:

```python
arr[0] = 99  # overwrite the first element
```

In JAX, that’s an error.

JAX arrays are **immutable**: once created, they can’t be changed.  
Instead, you create a *new* array with the change applied.

**Why should you care?** Immutability is the price of admission for the good stuff:
- **`jit`** can safely cache compiled code (because inputs won’t be mutated behind the scenes)
- **`grad`** can differentiate through your code reliably
- **parallelism** is easier because there’s no shared mutable state


If you’ve used Spark/Databricks, it’s similar to how DataFrames behave: you don’t “mutate” a DataFrame in place; you create a new one with a transformation.


In [4]:
# NumPy: in-place modification works
np_arr = np.array([1, 2, 3])
np_arr[0] = 99
print(f"NumPy after arr[0] = 99: {np_arr}")

# JAX: in-place modification raises an error
jax_arr = jnp.array([1, 2, 3])
try:
    jax_arr[0] = 99
except Exception as e:
    print(f"\nJAX error: {type(e).__name__} - can't modify in place!")

# JAX way: .at[].set() returns a NEW array
jax_arr_new = jax_arr.at[0].set(99)
print(f"\nOriginal JAX array: {jax_arr}   (unchanged!)")
print(f"New JAX array:      {jax_arr_new} (has the update)")

# Other .at[] operations you'll see:
# arr.at[i].add(val) - add to element i
# arr.at[mask].set(val) - set where mask is True
# arr.at[1:5].set(val) - set a slice

NumPy after arr[0] = 99: [99  2  3]

JAX error: TypeError - can't modify in place!

Original JAX array: [1 2 3]   (unchanged!)
New JAX array:      [99  2  3] (has the update)


### The Translation Cheat Sheet

| NumPy (what you'd write) | JAX (what to write instead) |
|:---|:---|
| `arr[i] = val` | `arr = arr.at[i].set(val)` |
| `arr[i] += val` | `arr = arr.at[i].add(val)` |
| `arr[mask] = val` | `arr = arr.at[mask].set(val)` |

That's the only syntax difference you need to internalize. Everything else (slicing, `reshape`, `concatenate`, `where`, broadcasting) works the same as NumPy.

## ⭐️📑 Concepts From the Paper
### `jit`
Traditional weather models are incredibly slow. For example:X-SHiELD needs nearly 14,000 computer cores to simulate just 19 days of weather in a 24-hour period. Because JAX data cannot be changed once created, the software can safely pre-compile complex physics equations into highly optimized code without worrying about unexpected data changes. This efficiency allows just one Google TPU processor to simulate 70,000 days of weather in that same 24 hours!

### `grad`
To learn effectively, NeuralGCM runs 5-day weather forecasts, calculating hundreds of sequential time steps. If the underlying data constantly overwrote itself along the way, the system would lose track of its mathematical steps. JAX's immutability preserves a flawless, step-by-step record. This allows the system to smoothly trace its forecast errors backward through time to properly train the neural network.

## parallelism
The biggest bottleneck in weather modeling is crunching massive grids of mathematical transformations. Because JAX data cannot be overwritten, there is no risk of different processors accidentally interfering with each other's work (aka **race conditions)**. This allows the software to seamlessly slice up the 3D global weather grid and calculate the heavy math simultaneously across multiple processors.

# Part 3: Quick Calculus Refresher

Before we look at `jax.grad()`, let's pull a core concept from our memory box: **a derivative measures change**.

## What's a derivative?

A derivative answers:

> "If I change an input by a tiny bit, how does the output change?"

**Concrete example (risk scoring):**

$$\text{ risk } = 0.5 \text{ amount }^2 + 3 \text{ amount }$$
$$\frac{d \text{ risk }}{d \text{ amount }} = \text{ amount } + 3$$

The derivative w.r.t `amount` tells you how much `risk` changes for a small change in `amount`.

> If we change the amount by this many units, how does risk change?

When `amount = 10`, the derivative is `13`, meaning that near amount = 10, increasing amount by 1 unit increases risk by about 13 units.

## What's a gradient?

In ML you rarely have one input - you have many inputs with **different weights**.

- A **derivative** is "rate of change w.r.t one number"
- A **gradient** is "a vector of rates of change" (one per parameter)

If your model has 10,000 weights, the gradient is a 10,000-length vector telling you which direction to adjust weights to reduce the loss.

Example: a model with many weights might look like:

$$
\hat{y} = w_1x_1 + w_2x_2 + w_3x_3 + \ldots + w_{10000}x_{10000} + b
$$

$$
\text{ loss } = ( \hat{y} - y)^2
$$


The gradient of `loss` is a long vector of partial derivatives (one partial derivative per parameter):

$$
\left[
\frac{\partial \text{loss}}{\partial w_1},\
\frac{\partial \text{loss}}{\partial w_2},\
\frac{\partial \text{loss}}{\partial w_3},\
\ldots,\
\frac{\partial \text{loss}}{\partial w_{10000}},\
\frac{\partial \text{loss}}{\partial b}
\right]
$$

This tells you how each weight should change to reduce the loss.

## Automatic differentiation and the chain rule (how JAX does this)

JAX uses **automatic differentiation (AD)** to compute derivatives.

### How automatic differentiation works
- JAX traces your function as a sequence of operations/primitives
- then applies differentiation rules to it, especially the **chain rule**

This is what people mean by "backprop": the chain rule applied many times through a computation.

Because JAX follows a functional programming style, you can also **stack `grad`** to get higher-order derivatives (derivatives of derivatives) when you need them.

## Why this matters for ML
Training is basically repeated:
1. compute loss
2. compute gradient of loss with respect to parameters
3. update parameters to reduce loss

Libraries like sklearn do this internally for specific models (like logistic regression).
JAX gives you the same mechanism, but you can point it at **any Python function made of JAX operations** - including custom simulations, custom losses, or hybrid physics+ML code.


## ⭐️📑 Concepts From the Paper

Because the researchers wrote their entire dynamical core ([dinosaur](https://github.com/neuralgcm/dinosaur)) natively in JAX, it became a fully traceable mathematical function. This allowed them to seamlessly merge it with a neural network to create the "first fully-differentiable hybrid general circulation model".

### 1. Compute Loss
Instead of making a single, isolated prediction, the model runs a continuous weather simulation for up to 5 days. At every single time step, it blends the hardcoded physics equations with the neural network's small-scale corrections. At the end of the simulation, it compares its final multi-day forecast against real-world historical weather data to see exactly how far off it was.

### 2. Compute Gradient of Loss with Respect to Parameters
Because the entire system is built in JAX, the software acts like a giant mathematical ***tracer***. It uses calculus to trace the final forecast error backward, step-by-step, through the entire multi-day sequence of physical and machine learning operations. The researchers call this "extended backpropagation". This step calculates exactly how a tiny change in the neural network's settings would have impacted the final forecast days later.


### 3. Update Parameters to Reduce Loss (Adjust & Learn)
Using that mapped-out mathematical trace, the system systematically tweaks the neural network's internal weights to improve the forecast for the next run.

### 💎⛏️
The reason this approach is so groundbreaking is that it forces the training process to account for the laws of physics. By training the neural network while it is actively coupled to the physics engine, it explicitly learns how its tiny, localized corrections will snowball and interact with large-scale weather patterns over several days. This eliminates the "blind spots" found in "offline-trained" models, preventing the mathematical instability and "climate drift" that used to ruin long-term simulations.


# Part 4: Transformation #1
## `jax.grad()`: Automatic Derivatives

`jax.grad(f)` takes a Python function `f` and returns a **new function** that computes the gradient of `f`.

In [5]:
# A simple function
def risk_score(amount):
    return 0.5 * amount**2 + 3 * amount

# jax.grad returns a NEW function that computes the gradient
risk_sensitivity = jax.grad(risk_score)

# Evaluate at amount = 10
amount = 10.0
print(f"Risk score at amount={amount}: {risk_score(amount)}")
print(f"Sensitivity (derivative) at amount={amount}: {risk_sensitivity(amount)}")
print(f"")
print(f"By hand: derivative of 0.5x² + 3x is x + 3 = 10 + 3 = 13.0")
print(f"JAX got: {risk_sensitivity(amount)}")
print(f"")
print(f"Interpretation: at amount=10, increasing the transaction by $1")
print(f"increases the risk score by ~{risk_sensitivity(amount):.0f} points.")

Risk score at amount=10.0: 80.0
Sensitivity (derivative) at amount=10.0: 13.0

By hand: derivative of 0.5x² + 3x is x + 3 = 10 + 3 = 13.0
JAX got: 13.0

Interpretation: at amount=10, increasing the transaction by $1
increases the risk score by ~13 points.


##`grad` Works on ***Any*** Function

This is where JAX gets powerful. It does not just handle simple formulas. It can differentiate through functions with loops, nested function calls, and fairly complex logic, as long as the computation is written using JAX-compatible operations. JAX traces through the computation and builds the derivative automatically.

In [6]:
# A more complex function with a loop
def compound_growth(rate, n_periods=12):
    """Compound growth over n periods, starting at $1."""
    value = 1.0
    for _ in range(n_periods):
        value = value * (1 + rate)
    return value

# How sensitive is the final value to the growth rate?
growth_sensitivity = jax.grad(compound_growth)

rate = 0.05  # 5% per period
print(f"Starting value: $1.00")
print(f"After 12 periods at {rate:.0%}: ${compound_growth(rate):.4f}")
print(f"Sensitivity to rate: {growth_sensitivity(rate):.4f}")
print(f"")
print(f"Meaning: a 1 percentage point increase in rate (5% -> 6%)")
print(f"would increase the final value by ~${growth_sensitivity(rate) * 0.01:.4f}")

Starting value: $1.00
After 12 periods at 5%: $1.7959
Sensitivity to rate: 20.5241

Meaning: a 1 percentage point increase in rate (5% -> 6%)
would increase the final value by ~$0.2052


## Gradients With Multiple Inputs

Most real functions have more than one input. Use `argnums` to pick which input(s) to differentiate with respect to.


$$\text{ Weighted Risk Score: } f(a,b) = a^2 + b^2 + 3ab$$
<br>

```grad_amount = jax.grad(weighted_score, argnums=0)```

$$\frac{\partial \text{ score }}{\partial \text{ amount }} = 2a +3b$$
<br>

```grad_freq = jax.grad(weighted_score, argnums=1)```

$$\frac{\partial \text{ score }}{\partial \text{ frequency }} = 2b +3a$$

In [7]:
def weighted_score(amount, frequency):
    """Risk score that depends on two features."""
    return amount**2 + frequency**2 + 3 * amount * frequency

# Derivative w.r.t 'amount' only (first argument, index 0)
grad_amount = jax.grad(weighted_score, argnums=0)
print(f"d(score)/d(amount) at (2, 3):    {grad_amount(2.0, 3.0)}")

# Derivative w.r.t 'frequency' only (second argument, index 1)
grad_freq = jax.grad(weighted_score, argnums=1)
print(f"d(score)/d(frequency) at (2, 3): {grad_freq(2.0, 3.0)}")

# Both at once
grad_both = jax.grad(weighted_score, argnums=(0, 1))
print(f"Both gradients at (2, 3):         {grad_both(2.0, 3.0)}")

d(score)/d(amount) at (2, 3):    13.0
d(score)/d(frequency) at (2, 3): 12.0
Both gradients at (2, 3):         (Array(13., dtype=float32, weak_type=True), Array(12., dtype=float32, weak_type=True))


### Rules for `jax.grad()`

1. **The function must return a scalar** (a single number, not an array). If your function returns an array, reduce it first with `jnp.mean(...)` or `jnp.sum(...)`.

2. **Inputs should be floats (or complex values).** JAX can allow integer inputs, but that is not the normal case.

3. **You can get the function value *and* gradient together** with `jax.value_and_grad()`. This is what you'll see in training loops because it avoids computing things twice:

```python
loss_val, grads = jax.value_and_grad(loss_fn)(params, data)
```

# Part 5: Transformation #2
## `jax.jit()`:Make It Fast

## The Problem

Python is slow. Every time you call a NumPy/JAX function, Python has to interpret the call, dispatch it, come back, interpret the next line, dispatch again, etc. For code that runs once, this is fine. For code you call thousands of times (like a training step), the overhead adds up.

## The Solution

`jax.jit(f)` **compiles** your function. The first time you call it, JAX inspects your code, converts it to optimized machine code, and caches the result. Every subsequent call runs the compiled version directly, no Python interpreter in the loop.

```
Without JIT:  Python → interpret line 1 → interpret line 2 → ... → result
With JIT:     Python → compiled blob → result   (first call is slow, rest are fast)
```

In [8]:
import time

# A function with a lot of repeated work
def compute_iteratively(x):
    for i in range(100):
        x = jnp.sin(x) + jnp.cos(x)
    return x

# JIT-compiled version
compute_fast = jax.jit(compute_iteratively)

x = jnp.ones(1000)

# First call triggers compilation (one-time cost)
print("Compiling...")
compute_fast(x).block_until_ready()  # block_until_ready: wait for GPU to finish
print("Done.\n")

# Benchmark: 10 calls each
start = time.perf_counter()
for _ in range(10):
    compute_iteratively(x).block_until_ready()
time_slow = time.perf_counter() - start

start = time.perf_counter()
for _ in range(10):
    compute_fast(x).block_until_ready()
time_fast = time.perf_counter() - start

print(f"Without JIT: {time_slow:.4f}s")
print(f"With JIT:    {time_fast:.4f}s")
print(f"Speedup:     {time_slow/time_fast:.0f}x")


Compiling...
Done.

Without JIT: 0.1043s
With JIT:    0.0125s
Speedup:     8x


## JIT Gotchas

JIT is powerful but opinionated. Here are the three issues you'll actually run into.

### Gotcha 1: Changing Array Shapes Triggers Recompilation

JIT compiles for a specific input shape. If the shape changes, it compiles a *new* version. This is fine occasionally, but if your data has variable-length rows, you'll see unexpected slowdowns.

**Analogy:** It's like Databricks query plan caching. If the plan changes, the cache can't help.

In [9]:
@jax.jit
def sum_array(x):
    return jnp.sum(x)

# Same shape = cached, fast
sum_array(jnp.ones(100))  # Compiles for shape (100,)
sum_array(jnp.ones(100))  # Uses cached version

# Different shape = recompiles
sum_array(jnp.ones(200))  # Compiles a new version for shape (200,)

print("Rule of thumb: keep array shapes consistent inside loops.")
print("Pad your data to a fixed size if needed (like padding sequences).")

Rule of thumb: keep array shapes consistent inside loops.
Pad your data to a fixed size if needed (like padding sequences).


### Gotcha 2: `print()` only runs during compilation, not execution

When JIT compiles your function, it first **traces** it once to build the computation graph.
Any Python `print()` inside a jitted function runs during that trace, then (usually) never again on later cached executions.

> A computation graph is like a depenency graph of computations.

**Use this instead:** `jax.debug.print()` - it prints at runtime *even under JIT*.

Also: prefer passing **JAX arrays** (e.g., `jnp.array(5.0)`) instead of bare Python floats.
Python scalars can cause accidental **re-tracing** when values change.


In [10]:
@jax.jit
def my_function(x):
    print(f"This runs during tracing: x = {x}")  # You'll see a Tracer object, not a number
    return x * 2

print("=== First call (triggers trace + compile) ===")
result1 = my_function(jnp.array(5.0))

print("\n=== Second call (runs compiled version - no print!) ===")
result2 = my_function(jnp.array(10.0))

print(f"\nResults: {result1}, {result2}")
print(f"")
print(f"For debugging inside JIT, use jax.debug.print() instead.")
print(f"Or remove @jax.jit temporarily while you're developing.")

=== First call (triggers trace + compile) ===
This runs during tracing: x = JitTracer<~float32[]>

=== Second call (runs compiled version - no print!) ===

Results: 10.0, 20.0

For debugging inside JIT, use jax.debug.print() instead.
Or remove @jax.jit temporarily while you're developing.


### Gotcha 3: Python `if` on JAX values doesn’t work inside `jit`

During tracing, JAX replaces your actual numbers with abstract **tracer** objects so it can build the computation graph.
Python’s `if` statement needs a concrete `True`/`False`, which a tracer can’t provide - so you’ll get a `TracerBoolConversionError`.

**What to use instead (pick the right tool):**
- `jnp.where(...)` for **elementwise** conditionals (arrays)
- `jax.lax.cond(...)` for **scalar branching** (true control flow)

If you know `np.where`, you already know `jnp.where`.


In [11]:
# BROKEN: Python 'if' on a traced value
@jax.jit
def apply_threshold_bad(x):
    if x > 0:        # Python needs a real bool - tracer can't provide one
        return x * 2
    else:
        return x * 3

# WORKS: jnp.where (same as np.where - you already know this one)
@jax.jit
def apply_threshold_good(x):
    return jnp.where(x > 0, x * 2, x * 3)

# Demo
print("Python 'if' inside JIT:")
try:
    apply_threshold_bad(jnp.array(5.0))
except Exception as e:
    print(f"  Error: {type(e).__name__}")

print(f"\njnp.where inside JIT:")
print(f"  threshold(5.0)  = {apply_threshold_good(jnp.array(5.0))}")
print(f"  threshold(-5.0) = {apply_threshold_good(jnp.array(-5.0))}")

# Quick reference:
# jnp.where(condition, a, b) - elementwise: pick a where True, b where False
# jax.lax.cond(pred, f, g, x) - scalar branch: run f(x) if pred, else g(x)

Python 'if' inside JIT:
  Error: TracerBoolConversionError

jnp.where inside JIT:
  threshold(5.0)  = 10.0
  threshold(-5.0) = -15.0


### When to use `jit` (and when not to)

**Use `jit` on:** functions you call many times with same-shaped inputs  
Examples: loss functions, simulation steps, per-batch prediction, training steps.

**Skip `jit` for:** data exploration, one-off computations, code you’re actively debugging, and anything with changing shapes each call.

**The standard pattern for this workshop:**
```python
# JIT the hot inner loop
@jax.jit
def train_step(params, batch):
    loss, grads = jax.value_and_grad(loss_fn)(params, batch)
    # ... optimizer update ...
    return params, loss

# Keep the outer loop in Python (logging, checkpoints, metrics)
for epoch in range(100):
    params, loss = train_step(params, batch)
    print(f"Epoch {epoch}: {loss}")  # fine because it's outside JIT
```

If you come from Spark/Databricks: think of `jit` as “compile the inner transform once, then run it fast many times.”


# Part 6: Transformation #3
## `jax.vmap()`: Automatic Batching

## The Problem

You write a function that works on one row. Now you need it to work on 10,000 rows. In Pandas you'd use `.apply()`. In NumPy you'd write a list comprehension or manually vectorize with broadcasting. Both have downsides. `.apply()` is slow, and manual vectorization is error-prone.

## The Solution

`jax.vmap(f)` is automatic vectorization: it takes a function written for **one input** and returns a new function that applies the same computation to **every element in a batch** automatically. Efficiently, without a Python loop.

In [12]:
# A function that scores ONE transaction
def score_transaction(features):
    """Simple risk scoring: weighted sum + nonlinearity."""
    weights = jnp.array([0.5, -0.3, 0.8])
    return jnp.tanh(jnp.dot(features, weights))

# Score a single transaction
single_txn = jnp.array([100.0, 3.0, 0.7])  # [amount, frequency, flag]
print(f"Single score: {score_transaction(single_txn):.4f}")

# Now score a BATCH of transactions
batch = jnp.array([
    [100.0, 3.0, 0.7],
    [500.0, 1.0, 0.2],
    [50.0,  8.0, 0.9],
])

# Option A: Python loop (slow, like df.apply(), each call goes through Python)
scores_loop = jnp.array([score_transaction(row) for row in batch])

# Option B: vmap (fast, JAX vectorizes into a single batched operation)
score_batch = jax.vmap(score_transaction)
scores_vmap = score_batch(batch)

print(f"\nLoop scores: {scores_loop}")
print(f"Vmap scores: {scores_vmap}")
print(f"Match: {jnp.allclose(scores_loop, scores_vmap)}")

Single score: 1.0000

Loop scores: [1. 1. 1.]
Vmap scores: [1. 1. 1.]
Match: True


## `in_axes`: Controlling What Gets Batched

Sometimes your function takes multiple arguments and you only want to batch over *some* of them. `in_axes` tells vmap which arguments to vectorize over (axis `0`) and which to keep fixed (`None`).

This is conceptually the same as:
```python
df.apply(lambda row: f(row, fixed_param), axis=1)
```
...except it runs at compiled speed instead of row-at-a-time Python speed.

In [13]:
def score_with_weights(features, weights):
    return jnp.tanh(jnp.dot(features, weights))

# Batch over features (axis 0), keep weights fixed (None)
score_batch = jax.vmap(score_with_weights, in_axes=(0, None))
#                                                   ^   ^
#                                                   |   |
#                                     Batch over rows   Same weights for every row

weights = jnp.array([0.5, -0.3, 0.8])
scores = score_batch(batch, weights)
print(f"Batch scores: {scores}")
print(f"")
print(f"in_axes=(0, None) means:")
print(f"  arg 0 (features): iterate over axis 0 (one row at a time)")
print(f"  arg 1 (weights):  don't iterate - pass the same value every time")

Batch scores: [1. 1. 1.]

in_axes=(0, None) means:
  arg 0 (features): iterate over axis 0 (one row at a time)
  arg 1 (weights):  don't iterate - pass the same value every time


# Part 7: Combining Transformations

JAX becomes most useful when you combine its core transformations. `grad`, `jit`, and `vmap` can be stacked to build fast, differentiable, batched computations.

This is the pattern you'll see in the main workshop:

```
grad  → compute the derivative of the loss function
vmap  → do it for every example in the batch
jit   → compile the whole thing for speed
```

In [14]:
# A simple loss function: squared prediction error
def loss(weights, features):
    prediction = jnp.dot(features, weights)
    return prediction**2  # Squared error (pretend target is 0)

# Step 1: grad - gradient of loss w.r.t. weights for one row of features
grad_loss = jax.grad(loss)

# Step 2: vmap - compute gradients for every row in a batch
# keep the first argument (weights) fixed; batch over axis 0 (rows) of the second argument (features)
grad_loss_batch = jax.vmap(grad_loss, in_axes=(None, 0))

# Step 3: jit - compile everything
grad_loss_batch_fast = jax.jit(grad_loss_batch)

# Try it
weights = jnp.array([1.0, 2.0, 3.0])
batch = jnp.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
])

grads = grad_loss_batch_fast(weights, batch)
print(f"Per-example gradients (3 examples x 3 weights):")
print(grads)
print(f"\nMean gradient (what you'd use to update weights): {jnp.mean(grads, axis=0)}")

Per-example gradients (3 examples x 3 weights):
[[ 28.  56.  84.]
 [256. 320. 384.]
 [700. 800. 900.]]

Mean gradient (what you'd use to update weights): [328. 392. 456.]


Conceptually,
```python
grad_loss_batch(weights, features_batch)
```

Is This
```python
[
    grad_loss(weights, features_batch[0]),
    grad_loss(weights, features_batch[1]),
    grad_loss(weights, features_batch[2]),
    ...
]
```

`jax.vmap(grad_loss, in_axes=(None, 0))` applies grad_loss across the rows of batch, because batch is the second argument and axis 0 means “iterate over rows.”

## The Training Loop You'll Use in the Main Workshop

This puts everything together using `optax`, a JAX optimization library. It plays a role similar to the optimizer inside tools like `sklearn.linear_model.SGDClassifier`, but in JAX you usually build the training step yourself and control each part directly.

In [17]:
import optax

# Initialize
weights = jnp.array([1.0, 2.0, 3.0])
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(weights)

# JIT-compiled training step
@jax.jit
def train_step(weights, opt_state, batch):
    # Compute loss and gradient in one pass
    def batch_loss(w):
        per_example = jax.vmap(lambda x: loss(w, x))(batch)
        return jnp.mean(per_example)

    loss_val, grads = jax.value_and_grad(batch_loss)(weights)

    # Optimizer turns gradients into weight updates
    updates, opt_state = optimizer.update(grads, opt_state, weights)
    weights = optax.apply_updates(weights, updates)

    return weights, opt_state, loss_val

# Training loop
for epoch in range(5):
    weights, opt_state, loss_val = train_step(weights, opt_state, batch)
    print(f"Epoch {epoch}: loss = {loss_val:.4f}, weights = {weights}")

print(f"\nWeights are approaching zero (our implicit target).")


Epoch 0: loss = 1240.0000, weights = [0.99000007 1.99       2.99      ]
Epoch 1: loss = 1228.2681, weights = [0.98000145 1.9800013  2.9800014 ]
Epoch 2: loss = 1216.5933, weights = [0.97000486 1.9700048  2.9700048 ]
Epoch 3: loss = 1204.9767, weights = [0.9600113 1.9600112 2.9600112]
Epoch 4: loss = 1193.4194, weights = [0.9500216 1.9500215 2.9500215]

Weights are approaching zero (our implicit target).


# Part 8: Random Numbers in JAX

## The Difference

In NumPy, random numbers come from a **stateful generator**. An object that silently advances its internal state each time you draw a number. This is convenient but can cause reproducibility headaches (especially in notebooks where cells get re-run out of order, or in parallel code).

JAX uses **explicit keys** instead. You pass a key to every random operation. Same key = same output, always. Think of it like setting a seed.

To get *different* random numbers, you **split** the key into new child keys.

It's more verbose, but your results are reproducible regardless of cell execution order or parallelism.

In [18]:
# NumPy: stateful, calling twice gives different values
rng = np.random.default_rng(42)
print("NumPy (stateful generator):")
print(f"  Call 1: {rng.standard_normal(3)}")
print(f"  Call 2: {rng.standard_normal(3)}  (different - internal state advanced)")

# JAX: explicit key,  same key always gives same output
key = jax.random.PRNGKey(42)
print(f"\nJAX (explicit key):")
print(f"  Call 1: {jax.random.normal(key, (3,))}")
print(f"  Call 2: {jax.random.normal(key, (3,))}  (identical - same key)")

# To get new values, split the key into children
key, subkey = jax.random.split(key)
print(f"\nAfter splitting:")
print(f"  New values: {jax.random.normal(subkey, (3,))}")


NumPy (stateful generator):
  Call 1: [ 0.30471708 -1.03998411  0.7504512 ]
  Call 2: [ 0.94056472 -1.95103519 -1.30217951]  (different - internal state advanced)

JAX (explicit key):
  Call 1: [-0.02830462  0.46713185  0.29570296]
  Call 2: [-0.02830462  0.46713185  0.29570296]  (identical - same key)

After splitting:
  New values: [ 0.60576403  0.7990441  -0.908927  ]


In [19]:
# The pattern you'll see in JAX codebases:

def my_random_function(key):
    """Accept a key, split internally, return the updated key."""
    key, k1, k2 = jax.random.split(key, 3)  # 1 key in, 3 keys out
    noise1 = jax.random.normal(k1, (5,))
    noise2 = jax.random.normal(k2, (5,))
    return noise1 + noise2, key  # Always return the updated key

# Thread the key through your calls
key = jax.random.PRNGKey(0)
result1, key = my_random_function(key)
result2, key = my_random_function(key)

print(f"Result 1: {result1}")
print(f"Result 2: {result2}")
print(f"\nDifferent results, but fully reproducible from the same starting key.")


Result 1: [-1.1468198  -0.68067    -0.20406133 -1.1254356  -0.38103232]
Result 2: [-2.001323    0.23587376 -3.4234405   0.6789133   0.04178387]

Different results, but fully reproducible from the same starting key.


# Part 9: Where JAX Fits - A Practical Decision Guide

| Situation | Reach for... |
|:----------|:-------------|
| Loading/cleaning tabular data | **Pandas** (or PySpark/SQL for big data) |
| Quick exploratory array math | **NumPy** |
| Standard ML: classification, regression, clustering | **sklearn, xgboost, lightgbm** |
| Custom loss functions that need derivatives | **JAX** |
| Speeding up a NumPy bottleneck with compilation | **JAX** |
| Simulations coupled with ML optimization | **JAX** |
| Large-scale deep learning | **PyTorch** or **JAX + Flax** |

For this workshop, JAX lets us write a physics simulation and an ML model in the same framework, then compute derivatives through *both at once*. That's the capability that makes hybrid physics-ML models practical.

## ⭐️📑 Concepts From the Paper

To get the AI and the physics engine to work together, the researchers couldn't just attach a neural network to an existing weather simulator. Traditional weather models (like NOAA's FV3, which relies on over 376,000 lines of code) are built using older programming languages like Fortran. These older systems are completely incompatible with "automatic differentiation".

### Dinosaur 🦕
To solve this, the researchers built a brand new, open-source physics engine from scratch called **Dinosaur**. **Dinosaur** is essentially a classic, "old-fashioned" weather physics simulator that has been completely rewritten in JAX.

### The Breakthrough
Because Dinosaur is written in JAX, it n**atively supports automatic differentiation**. This allows the physics engine and the neural network to seamlessly merge into one unified system, enabling a breakthrough process called "online training".

* **The Physics Engine (Dynamical Core):** This part handles the "big picture." It uses well-established laws of physics and standard mathematical equations to calculate massive, global weather patterns, like large-scale wind currents and major temperature shifts.

* **The AI (Neural Network):** This part handles the small-scale, chaotic details. Because phenomena like localized cloud formation, rainfall, and sunlight radiation are simply too complex and random to capture with rigid math equations, the neural network steps in to predict these highly localized events by recognizing patterns.






# Cheat Sheet

Save this for later because we'll use it during the main workshop.

## The Three Transformations

| Transformation | What it does | When to use it |
|:---------------|:-------------|:---------------|
| `jax.grad(f)` | Returns a function that computes derivatives of `f` | Optimization, sensitivity analysis, training |
| `jax.jit(f)` | Compiles `f` to fast machine code | Functions called many times with fixed shapes |
| `jax.vmap(f)` | Auto-vectorizes `f` over a batch dimension | Applying a per-row function to an array |

## Recipes

```python
# Loss + gradient in one pass
loss_val, grads = jax.value_and_grad(loss_fn)(params, data)

# Vectorize: per-row function → batch function
batched_fn = jax.vmap(row_fn, in_axes=(0, None))  # batch arg0, broadcast arg1

# Compile for speed
fast_fn = jax.jit(my_fn)   # or use @jax.jit as a decorator

# Array update (immutable - returns a new array)
new_arr = arr.at[i].set(val)

# Random numbers
key, subkey = jax.random.split(key)
noise = jax.random.normal(subkey, shape)
```

## Troubleshooting

| Symptom | Cause | Fix |
|:--------|:------|:----|
| `TypeError: JAX arrays are immutable` | You wrote `arr[i] = val` | Use `arr = arr.at[i].set(val)` |
| First call slow, rest fast | JIT compilation (one-time) | Normal - warm up before benchmarking |
| JIT recompiles every call | Input shapes keep changing | Keep shapes consistent; pad if needed |
| `print()` shows `Traced<...>` | Printing inside a JIT-compiled function | Use `jax.debug.print()` or remove `@jit` |
| Same random numbers every time | Reusing the same key | Split the key: `key, subkey = jax.random.split(key)` |
| `ConcretizationTypeError` on `if` | Python `if` on a traced value inside JIT | Use `jnp.where()` or `jax.lax.cond()` |
| `grad` errors on integer input | JAX can't differentiate integers | Use `10.0` (float) instead of `10` (int) |

## You're Ready For The Main Workshop 🥳

Here's what to keep in mind:

- JAX arrays work like NumPy, with one rule: no in-place mutation (use `.at[].set()`).
- `jax.grad()` computes derivatives of any function. The computer does the calculus for you.:
- `jax.jit()` compiles functions for speed. Use on hot loops, skip during debugging.
- `jax.vmap()` auto-vectorizes a per-row function over a batch. Think `df.apply()` but fast.
- Random numbers need explicit keys. Always split.

If something in the main workshop doesn't make sense, come back here as a reference.

**Want to go deeper?** The [JAX quickstart](https://jax.readthedocs.io/en/latest/notebooks/quickstart.html) and [JAX 101 tutorial](https://jax.readthedocs.io/en/latest/jax-101/index.html) are both well-written and practical.